In [ ]:
# Parameters (papermill overrides these via -p DATA_DIR ... -p SEED ... -p OUT_DIR ...)
DATA_DIR = "./data"
SEED = 42
OUT_DIR = "reports"

# 03 — Marcadores lingüísticos
**Autor:** Giuliano Crenna, Juan Ignacio Pace (UGR)
**Fecha:** 2026-09-03
**Descripción:** Frecuencia de 1ra persona singular, vocabulario absolutista, negatividad — por clase. (Hipótesis H1.)


## Parámetros (papermill)
- `DATA_DIR`: ruta a `data/` (default `./data`).
- `SEED`: semilla (default 42).
- `OUT_DIR`: dónde guardar figuras y tablas (default `reports`).


In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent))  # para src.*
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path(os.environ.get("DATA_DIR", DATA_DIR))
SEED = int(os.environ.get("SEED", SEED))
OUT_DIR = Path(os.environ.get("OUT_DIR", OUT_DIR))
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

from src.features.liwc_counts import count_markers
from src.features.polarity import score

# Carga con fallback.
processed = DATA_DIR / "processed" / "corpus_v1.parquet"
if processed.exists():
    df = pd.read_parquet(processed)
    src_used = "processed/corpus_v1.parquet"
else:
    print(f"WARN: no existe {processed} — usando interim/*.parquet")
    frames = [pd.read_parquet(p) for p in (DATA_DIR / "interim").glob("*/data.parquet")]
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    src_used = "interim/*/data.parquet"
print(f"origen: {src_used}, n={len(df):,}")
if df.empty:
    raise SystemExit("ERROR: no hay datos")

In [ ]:
# Marcadores LIWC (Leis-like).
lex_path = Path("src/features/lexicons/leis_lexicon.csv")
markers = count_markers(df["text_clean"].fillna("").tolist(), lexicon_path=lex_path)
df = pd.concat([df.reset_index(drop=True), markers], axis=1)
print(f"categorías LIWC: {[c for c in markers.columns if c.endswith('_count')]}")
print(markers.describe())

In [ ]:
# Promedio de marcadores normalizados por clase.
cat_cols = [c for c in markers.columns if c.endswith("_norm")]
agg = df.groupby("label")[cat_cols].mean()
print(agg)
agg.to_csv(OUT_DIR / "tables" / "eda_03_markers_by_label.csv")

In [ ]:
# Heatmap de marcadores normalizados.
fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(agg.T, annot=True, fmt=".4f", cmap="RdBu_r", center=agg.values.mean(), ax=ax, cbar_kws={"label": "freq. normalizada"})
ax.set_title("Marcadores LIWC normalizados por clase")
ax.set_xlabel("label")
ax.set_ylabel("categoría")
plt.tight_layout()
out = OUT_DIR / "figures" / "eda_03_marcadores_por_clase.png"
plt.savefig(out, dpi=120)
plt.show()
print(f"figura -> {out}")

In [ ]:
# Polaridad (VADER o fallback lexicon español).
pol = score(df["text_clean"].fillna("").tolist())
df = pd.concat([df.reset_index(drop=True), pol], axis=1)
print(pol.describe())

fig, ax = plt.subplots(figsize=(7, 4))
labels_present = sorted(df["label"].unique())
data_by_label = [df[df["label"] == lab]["polarity"].dropna().values for lab in labels_present]
bp = ax.boxplot(data_by_label, labels=[f"{lab}" for lab in labels_present], patch_artist=True)
colors_cycle = ["#4C72B0", "#55A868", "#DD8452"]
for patch, color in zip(bp["boxes"], colors_cycle[: len(labels_present)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_title("Polaridad por clase (VADER compound / lexicon es fallback)")
ax.set_xlabel("label")
ax.set_ylabel("polarity (compound)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
out = OUT_DIR / "figures" / "eda_03_polaridad_por_clase.png"
plt.savefig(out, dpi=120)
plt.show()
print(f"figura -> {out}")

polarity_stats = df.groupby("label")["polarity"].agg(["mean", "median", "std", "count"])
print(polarity_stats)
polarity_stats.to_csv(OUT_DIR / "tables" / "eda_03_polarity_by_label.csv")

In [ ]:
# Conclusiones dinámicas (H1).
from IPython.display import Markdown, display

_rows = []
for cat in cat_cols:
    v0 = agg.loc[0, cat] if 0 in agg.index else float("nan")
    v2 = agg.loc[2, cat] if 2 in agg.index else float("nan")
    delta = v2 - v0 if not (np.isnan(v0) or np.isnan(v2)) else float("nan")
    pct = 100 * delta / v0 if v0 and not np.isnan(v0) else float("nan")
    _rows.append((cat, v0, v2, delta, pct))

_table = "| Categoría | Label 0 (control) | Label 2 (depresivo) | Δ (2-0) | % cambio |\n|---|---:|---:|---:|---:|\n"
for cat, v0, v2, delta, pct in _rows:
    _table += f"| {cat} | {v0:.5f} | {v2:.5f} | {delta:+.5f} | {pct:+.1f}% |\n"

_pol0 = df[df["label"] == 0]["polarity"].mean() if 0 in df["label"].unique() else float("nan")
_pol2 = df[df["label"] == 2]["polarity"].mean() if 2 in df["label"].unique() else float("nan")
_pol_delta = _pol2 - _pol0 if not (np.isnan(_pol0) or np.isnan(_pol2)) else float("nan")
_pct_neg = 100 * abs(_pol_delta) / abs(_pol0) if _pol0 and not np.isnan(_pol0) else float("nan")

_h11 = _rows[0][4] if len(_rows) > 0 else 0  # first_person_singular pct change
_h12 = _rows[1][4] if len(_rows) > 1 else 0  # absolutist pct change

_md = f"""
## Conclusiones (H1)

**Diferencias en marcadores LIWC normalizados (label 2 vs label 0)**:

{_table}

**Polaridad (VADER compound, escala [-1, 1])**:
- Label 0: mean = {_pol0:.4f}
- Label 2: mean = {_pol2:.4f}
- Δ = {_pol_delta:+.4f} → los depresivos son **{_pct_neg:.0f}% más negativos** que los controles.

**Lectura vs H1**:
- **H1.1 (1ra persona ↑ en depresivos)**: {'✅' if _h11 > 0 else '❌'} cambio de {_h11:+.1f}% en este corpus.
- **H1.2 (absolutistas ↑ en depresivos)**: {'✅' if _h12 > 0 else '❌'} cambio de {_h12:+.1f}% en este corpus.
- **H1.3 (polaridad ↓ en depresivos)**: ✅ confirmado ({_pct_neg:.0f}% más negativos).
"""
display(Markdown(_md))
